# 10 — KYC Compliance & AML Risk Prediction
Predictive modeling for KYC risk based on client profiles, sanctions, and aggregated transaction behavior.

In [1]:
import pandas as pd, numpy as np, sqlite3
import plotly.express as px, plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
import joblib, os
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

In [2]:
# ── Load and Target Engineering ──
df = pd.read_csv('../../data/processed/kyc_clean.csv')
print(f"Loaded: {df.shape}")
print(f"Columns: {list(df.columns)}")

df['kyc_risk_flag'] = (
    (df['pep_flag'] == 1) |
    (df['sanctions_flag'] == 1) |
    (df['fatf_country_flag'] == 1) |
    (df.get('ofac_matches', pd.Series(0, index=df.index)).fillna(0) > 0) |
    (df.get('structuring_flags', pd.Series(0, index=df.index)).fillna(0) > 2) |
    (df['ownership_opacity_score'] > 0.3)
).astype(int)

print(f"\nKYC risk flag distribution:")
print(df['kyc_risk_flag'].value_counts())
print(f"Risk rate: {df['kyc_risk_flag'].mean()*100:.2f}%")

sector_map = {'Low': 0, 'Medium': 1, 'High': 2}
df['sector_risk_enc'] = df['sector_risk'].map(sector_map).fillna(1)

Loaded: (2000, 24)
Columns: ['client_id', 'client_name', 'client_type', 'sector', 'sector_risk', 'country', 'pep_flag', 'sanctions_flag', 'fatf_country_flag', 'ofac_country_flag', 'sectoral_sanctions_flag', 'ownership_opacity_score', 'total_transactions', 'total_amount', 'avg_amount', 'ofac_matches', 'fatf_txn_flags', 'structuring_flags', 'rapid_movement_flags', 'trade_mispricing_flags', 'unique_counterparty_countries', 'client_country', 'composite_risk_score', 'risk_tier']

KYC risk flag distribution:
kyc_risk_flag
1    1032
0     968
Name: count, dtype: int64
Risk rate: 51.60%


In [3]:
# ── Feature Definition ──
feature_cols = [
    'pep_flag', 'sanctions_flag', 'fatf_country_flag',
    'ofac_country_flag', 'sectoral_sanctions_flag',
    'ownership_opacity_score', 'sector_risk_enc'
]

txn_features = ['ofac_matches','structuring_flags','rapid_movement_flags',
                'fatf_txn_flags','total_transactions','avg_amount',
                'unique_counterparty_countries']
for f in txn_features:
    if f in df.columns:
        feature_cols.append(f)
        df[f] = df[f].fillna(0)

print(f"Using {len(feature_cols)} features: {feature_cols}")
X = df[feature_cols]
y = df['kyc_risk_flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Using 14 features: ['pep_flag', 'sanctions_flag', 'fatf_country_flag', 'ofac_country_flag', 'sectoral_sanctions_flag', 'ownership_opacity_score', 'sector_risk_enc', 'ofac_matches', 'structuring_flags', 'rapid_movement_flags', 'fatf_txn_flags', 'total_transactions', 'avg_amount', 'unique_counterparty_countries']
Train: (1600, 14), Test: (400, 14)


In [4]:
# ── Train Models ──
pos_w = (y_train==0).sum() / max((y_train==1).sum(), 1)

rf_kyc = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf_kyc.fit(X_train, y_train)
rf_prob = rf_kyc.predict_proba(X_test)[:,1]
rf_pred = rf_kyc.predict(X_test)

xgb_kyc = XGBClassifier(scale_pos_weight=pos_w, random_state=42, verbosity=0, n_estimators=200)
xgb_kyc.fit(X_train, y_train)
xgb_prob = xgb_kyc.predict_proba(X_test)[:,1]
xgb_pred = xgb_kyc.predict(X_test)

print("=== KYC MODEL RESULTS (REAL) ===")
for name, prob, pred in [
    ('Random Forest', rf_prob, rf_pred),
    ('XGBoost', xgb_prob, xgb_pred)
]:
    print(f"\n{name}:")
    print(f"  Accuracy: {accuracy_score(y_test, pred):.4f}")
    print(f"  F1: {f1_score(y_test, pred):.4f}")
    print(f"  ROC-AUC: {roc_auc_score(y_test, prob):.4f}")
    print(classification_report(y_test, pred, target_names=['Low Risk','High Risk']))

=== KYC MODEL RESULTS (REAL) ===

Random Forest:
  Accuracy: 1.0000
  F1: 1.0000
  ROC-AUC: 1.0000
              precision    recall  f1-score   support

    Low Risk       1.00      1.00      1.00       194
   High Risk       1.00      1.00      1.00       206

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400


XGBoost:
  Accuracy: 1.0000
  F1: 1.0000
  ROC-AUC: 1.0000
              precision    recall  f1-score   support

    Low Risk       1.00      1.00      1.00       194
   High Risk       1.00      1.00      1.00       206

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



In [5]:
# ── Risk Tier Assignment ──
df['risk_score'] = xgb_kyc.predict_proba(X)[:,1]

df['risk_level'] = pd.cut(
    df['risk_score'],
    bins=[-0.001, 0.25, 0.50, 0.75, 1.001],
    labels=['Low Risk', 'Medium Risk', 'High Risk', 'Critical']
)

def get_action(level):
    actions = {
        'Low Risk': 'No action required',
        'Medium Risk': 'Send email reminder',
        'High Risk': 'Phone call required',
        'Critical': 'Compliance team intervention'
    }
    return actions.get(str(level), 'Review required')

df['recommended_action'] = df['risk_level'].apply(get_action)

print("\n=== REAL RISK TIER DISTRIBUTION ===")
print(df['risk_level'].value_counts())
print(df.groupby('risk_level', observed=True)['risk_score'].agg(['min','mean','max']).round(4))


=== REAL RISK TIER DISTRIBUTION ===
risk_level
Critical       1032
Low Risk        968
Medium Risk       0
High Risk         0
Name: count, dtype: int64
              min   mean    max
risk_level                     
Low Risk   0.0010 0.0036 0.0112
Critical   0.9466 0.9963 0.9997


In [6]:
# ── SQL for Compliance Reporting ──
con = sqlite3.connect(':memory:')
df.to_sql('kyc', con, index=False, if_exists='replace')

q1 = pd.read_sql_query("""
    SELECT sector, sector_risk, risk_level, COUNT(*) as count,
           ROUND(AVG(risk_score),4) as avg_score
    FROM kyc GROUP BY sector, sector_risk, risk_level ORDER BY avg_score DESC
""", con)

q2 = pd.read_sql_query("""
    SELECT pep_flag, sanctions_flag, COUNT(*) as count,
           ROUND(AVG(risk_score),4) as avg_risk_score,
           SUM(CASE WHEN risk_level IN ('High Risk','Critical') THEN 1 ELSE 0 END) as high_risk_count
    FROM kyc GROUP BY pep_flag, sanctions_flag ORDER BY avg_risk_score DESC
""", con)

q3 = pd.read_sql_query("""
    SELECT country, COUNT(*) as clients,
           ROUND(AVG(risk_score),4) as avg_risk,
           SUM(CASE WHEN risk_level='Critical' THEN 1 ELSE 0 END) as critical_count
    FROM kyc GROUP BY country ORDER BY avg_risk DESC LIMIT 20
""", con)

In [7]:
# ── Visualizations ──
# Plot 1: Risk tier distribution
fig1 = px.pie(df, names='risk_level', title='KYC Risk Tier Distribution', hole=0.4)
fig1.show()

# Plot 2: ROC curves
fig2 = go.Figure()
for name, prob in [('Random Forest', rf_prob), ('XGBoost', xgb_prob)]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    fig2.add_trace(go.Scatter(x=fpr, y=tpr, name=name))
fig2.add_trace(go.Scatter(x=[0, 1], y=[0, 1], name="Baseline", line=dict(color='black', dash='dash')))
fig2.update_layout(title="ROC Curves — KYC Risk Prediction", template='plotly_white')
fig2.show()

# Plot 3: Feature importance
xgb_fi = pd.Series(xgb_kyc.feature_importances_, index=feature_cols).sort_values()
fig3 = px.bar(x=xgb_fi.values, y=xgb_fi.index, orientation='h', title="Feature Importance (XGBoost)")
fig3.show()

# Plot 4: Risk score distribution by sector_risk
fig4 = px.violin(df, x='sector_risk', y='risk_score', color='sector_risk', box=True, title="Risk Score Distribution by Sector Risk")
fig4.show()

# Plot 5: Country risk heatmap
fig5 = px.bar(q3, x='country', y='avg_risk', color='critical_count', title="Average KYC Risk Score by Country")
fig5.show()

# Plot 6: PEP x Sanctions matrix
q2_pivot = q2.pivot(index='pep_flag', columns='sanctions_flag', values='avg_risk_score').fillna(0)
fig6 = px.imshow(q2_pivot, text_auto=True, title="PEP x Sanctions: Avg Risk Score")
fig6.show()

# Plot 7: Confusion matrix
cm = confusion_matrix(y_test, xgb_pred)
fig7 = px.imshow(cm, text_auto=True, title="XGBoost Confusion Matrix", labels=dict(x="Predicted", y="Actual"), x=['Low Risk','High Risk'], y=['Low Risk','High Risk'])
fig7.show()

# Plot 8: Risk score by client_type
fig8 = px.box(df, x='client_type', y='risk_score', color='client_type', title="Risk Score by Client Type")
fig8.show()

In [8]:
# ── Save ──
os.makedirs('../../models/kyc', exist_ok=True)
os.makedirs('../../data/features', exist_ok=True)
joblib.dump(rf_kyc, '../../models/kyc/random_forest_kyc.pkl')
joblib.dump(xgb_kyc, '../../models/kyc/xgboost_kyc.pkl')
df.to_csv('../../data/processed/kyc_with_risk.csv', index=False)
df[['client_id','risk_score','risk_level','recommended_action']].to_csv('../../data/features/kyc_features.csv', index=False)
print("=== SAVED. Real risk distribution:")
print(df['risk_level'].value_counts().to_string())
con.close()

=== SAVED. Real risk distribution:
risk_level
Critical       1032
Low Risk        968
Medium Risk       0
High Risk         0
